# ERA5 to PRISM Downscaling (Georgia)
Regional spatiotemporal downscaling with baseline-to-temporal model comparison.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not (ROOT / 'datasets').exists() and (ROOT.parent / 'datasets').exists():
    ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))


## 1. Problem Setup
ERA5 is coarse-resolution reanalysis data. PRISM is higher-resolution climate data.
The objective is to learn a mapping from ERA5 sequence inputs to PRISM target grids.


## 2. Modeling Approach
Models used in this pipeline:
- persistence baseline
- linear baseline
- CNN as spatial baseline
- ConvLSTM as temporal model

The purpose is to test whether temporal context improves downscaling performance.


### Why this setup is useful
- real climate systems depend on both spatial and temporal information
- downscaling is necessary for regional-level analysis
- this pipeline provides a controlled setup to test spatiotemporal learning behavior


## 3. Data Overview
ERA5 is organized by time, latitude, and longitude in NetCDF format. PRISM is provided as dated raster grids.
Temporal windows are formed as `ERA5(t-k+1 ... t)` and matched to `PRISM(t)` by date.


In [ ]:
import json
import pandas as pd
import xarray as xr
import rioxarray
from IPython.display import Image, Markdown, display

ERA5_PATH = ROOT / 'data_raw/era5_georgia_temp.nc'
PRISM_DIR = ROOT / 'data_raw/prism'
RESULTS_DIR = ROOT / 'results/evaluation'

print(f'Project root: {ROOT}')
print(f'ERA5 path: {ERA5_PATH}')
print(f'PRISM path: {PRISM_DIR}')
print(f'Results path: {RESULTS_DIR}')
print(f'ERA5 available: {ERA5_PATH.exists()}')
print(f'PRISM available: {PRISM_DIR.exists()}')


In [ ]:
if ERA5_PATH.exists() and PRISM_DIR.exists():
    try:
        era5_ds = xr.open_dataset(ERA5_PATH)
        print('ERA5 variables:', list(era5_ds.data_vars))
        if era5_ds.data_vars:
            var = list(era5_ds.data_vars)[0]
            print(f'ERA5 {var} dims:', era5_ds[var].dims)
            print(f'ERA5 {var} shape:', tuple(era5_ds[var].shape))

        prism_files = sorted([p for p in PRISM_DIR.rglob('*') if p.suffix.lower() in {'.bil', '.tif', '.tiff', '.asc'}])
        print('PRISM raster count:', len(prism_files))
        if prism_files:
            prism_da = rioxarray.open_rasterio(prism_files[0], masked=True)
            print('Sample PRISM raster:', prism_files[0].name)
            print('PRISM dims:', prism_da.dims)
            print('PRISM shape:', tuple(prism_da.shape))

        from datasets.prism_dataset import ERA5_PRISM_Dataset
        ds = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_DIR), history_length=5, verbose=False)
        x, y = ds[0]
        print(f'Usable samples: {len(ds)}')
        print(f'Input shape [T, C, H, W]: {tuple(x.shape)}')
        print(f'Target shape [C, H, W]: {tuple(y.shape)}')
    except Exception:
        print('Data inspection could not be completed with current files.')
        print('Run:')
        print('python data_pipeline/download_era5_georgia.py --year 2023 --month 1')
        print('python data_pipeline/download_prism.py --start-date 20230101 --days 20 --variable tmean')
else:
    print('Data files are missing.')
    print('Run:')
    print('python data_pipeline/download_era5_georgia.py --year 2023 --month 1')
    print('python data_pipeline/download_prism.py --start-date 20230101 --days 20 --variable tmean')


## 4. Model Comparison
Persistence is the low-complexity reference. The linear baseline adds a global correction. CNN learns spatial mapping at each timestep. ConvLSTM uses sequence context directly to model temporal structure.


## 5. Results


### Model Comparison Summary
- linear model performs best with current data
- deep models show expected behavior but need more data


In [ ]:
cnn_images = sorted((RESULTS_DIR / 'cnn').glob('comparison_*.png'))
convlstm_images = sorted((RESULTS_DIR / 'convlstm').glob('comparison_*.png'))
cnn_metrics_path = RESULTS_DIR / 'cnn' / 'metrics.json'
convlstm_metrics_path = RESULTS_DIR / 'convlstm' / 'metrics.json'
summary_csv_path = RESULTS_DIR / 'baselines_summary.csv'
model_comparison_path = RESULTS_DIR / 'model_comparison.png'

if cnn_images:
    display(Markdown('### CNN output'))
    display(Image(filename=str(cnn_images[0])))
else:
    display(Markdown('CNN output not found.'))

if convlstm_images:
    display(Markdown('### ConvLSTM output'))
    display(Image(filename=str(convlstm_images[0])))
else:
    display(Markdown('ConvLSTM output not found.'))

if model_comparison_path.exists():
    display(Markdown('### Model comparison figure'))
    display(Image(filename=str(model_comparison_path)))
else:
    display(Markdown('Model comparison figure not found.'))

rows = []
for model_name, path in [('cnn', cnn_metrics_path), ('convlstm', convlstm_metrics_path)]:
    if path.exists():
        data = json.loads(path.read_text())
        rows.append({
            'model': model_name,
            'rmse': data.get('rmse'),
            'mae': data.get('mae'),
            'bias': data.get('bias'),
            'num_samples': data.get('num_samples'),
        })

if rows:
    metrics_df = pd.DataFrame(rows)
    display(Markdown('### Metrics from JSON'))
    display(metrics_df)
else:
    display(Markdown('Metrics JSON files not found.'))

if summary_csv_path.exists():
    display(Markdown('### Baseline summary table'))
    display(pd.read_csv(summary_csv_path))
else:
    display(Markdown('baselines_summary.csv not found.'))

missing_any = (
    (not cnn_images)
    or (not convlstm_images)
    or (not cnn_metrics_path.exists())
    or (not convlstm_metrics_path.exists())
    or (not summary_csv_path.exists())
    or (not model_comparison_path.exists())
)
if missing_any:
    print('Run:')
    print('python training/train_downscaler.py --model cnn --history-length 1')
    print('python training/train_downscaler.py --model convlstm --history-length 3')
    print('python evaluation/evaluate_model.py --history-length 3')


## 6. Metrics Interpretation
RMSE and MAE measure average temperature error; lower values indicate better agreement with PRISM.
With limited temporal coverage, comparisons between CNN and ConvLSTM should be treated as preliminary.


## 7. Visual Analysis
- ERA5 inputs are smooth because of lower spatial resolution
- PRISM targets contain finer spatial structure
- CNN partially reconstructs structure but smooths details
- ConvLSTM improves consistency across time but still lacks resolution fidelity


## 8. Limitations
- very small temporal dataset
- single variable (temperature only)
- no multi-source inputs
- models are small-scale compared with modern large systems


### Relation to Existing Work
- ConvLSTM is a standard approach for spatiotemporal modeling
- modern weather systems rely heavily on temporal context
- this project focuses on a smaller regional setup rather than large-scale models


## 9. Future Work
- extend temporal window (longer sequences)
- include additional ERA5 variables (wind, pressure, humidity)
- incorporate multi-source inputs (e.g., remote sensing or high-resolution imagery)
- explore uncertainty-aware models for forecasting


### Key Findings
- Linear baseline currently performs best under limited data conditions
- CNN captures spatial structure but smooths high-resolution details
- ConvLSTM introduces temporal consistency but is constrained by dataset size


### What this indicates
- the pipeline is functioning correctly
- performance is limited by data, not model design
- temporal models require longer sequences to be effective


### Conclusion
- pipeline is working correctly
- results reflect data limitations, not model failure


### Direction for Extension
This setup is designed to extend toward:

- longer temporal context (multi-step ERA5 inputs)
- multi-variable conditioning (temperature, wind, pressure)
- multi-source data integration (e.g., remote sensing or high-resolution imagery)
- uncertainty-aware prediction when moving toward forecasting

The current implementation focuses on validating the modeling pipeline before scaling to higher-dimensional inputs.
